# Day 11 — GANs & VAEs: Image Augmentation | RetailMind AI

**Objective:** Generate synthetic fashion-product images for an under-represented category and prepare reviewed images for the RetailMind vision classifier.

This increment uses a **Deep Convolutional GAN (DCGAN)**:
- Generator: maps random noise to a 64×64 RGB product image.
- Discriminator: distinguishes real product images from generated images.
- Training: adversarial learning with `GradientTape`.
- Output: generated images saved to a review folder, plus a reusable generator model.

### Dataset
Download [Fashion Product Images (Small)](https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small) and place/extract it under:

```text
RetailMind/
├── data/
│   └── fashion-product-images-small/
│       ├── images/
│       └── styles.csv
├── models/
├── notebooks/
│   └── day11_gan_image_augmentation.ipynb
└── ...
```

If your extracted folder differs, update `DATA_ROOT` in the configuration cell.

> **Important:** GAN output must be visually reviewed. This notebook saves generated images to a review folder first; only manually approved images should be copied into the classifier's training data. Synthetic images must never be added to the test set.

## 1. Imports and configuration

TensorFlow/Keras is used for image loading, model construction, training, and image export. Python standard-library modules are used for paths and CSV metadata. Matplotlib is used for visual inspection.

In [ ]:
from pathlib import Path
import csv
import re
import tensorflow as tf
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

DATA_ROOT = ROOT_DIR / "data" / "fashion-product-images-small"
IMAGE_DIR = DATA_ROOT / "images"
STYLES_CSV = DATA_ROOT / "styles.csv"

MODEL_DIR = ROOT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_ROOT = ROOT_DIR / "data" / "synthetic_fashion"
REVIEW_DIR = OUTPUT_ROOT / "review"
APPROVED_DIR = OUTPUT_ROOT / "approved"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
APPROVED_DIR.mkdir(parents=True, exist_ok=True)

# Image / GAN settings
IMAGE_SIZE = 64
CHANNELS = 3
LATENT_DIM = 128
BATCH_SIZE = 64
EPOCHS = 100
MAX_IMAGES = 5000          # Cap training images to keep this increment manageable.
NUM_IMAGES_TO_GENERATE = 200
RANDOM_SEED = 42

# Choose the category field available in styles.csv:
# "articleType" gives fine-grained classes; "masterCategory" gives broad classes.
CATEGORY_COLUMN = "articleType"

# Optional: path to a Phase-2 confusion matrix CSV.
# Expected format: first column = true class labels; remaining columns = predicted class labels.
CONFUSION_MATRIX_PATH = ROOT_DIR / "reports" / "phase2_confusion_matrix.csv"

tf.keras.utils.set_random_seed(RANDOM_SEED)

print("Project root:", ROOT_DIR)
print("Image directory exists:", IMAGE_DIR.exists())
print("Styles CSV exists:", STYLES_CSV.exists())

## 2. Read metadata and inspect category imbalance

The dataset's `styles.csv` typically contains `id`, `masterCategory`, `subCategory`, and `articleType`. We count images by the configured category field.

If you have the Phase-2 confusion matrix, load it in the next cell. The lowest-recall class is a useful candidate for augmentation, but inspect its support as well: a class with almost no images may be too small for a GAN to learn reliably.

In [ ]:
if not STYLES_CSV.exists():
    raise FileNotFoundError(
        f"Could not find {STYLES_CSV}. Check DATA_ROOT and dataset extraction."
    )

metadata_rows = []
with STYLES_CSV.open("r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    if CATEGORY_COLUMN not in (reader.fieldnames or []):
        raise ValueError(
            f"'{CATEGORY_COLUMN}' not found in styles.csv. "
            f"Available columns: {reader.fieldnames}"
        )

    for row in reader:
        image_id = (row.get("id") or "").strip()
        category = (row.get(CATEGORY_COLUMN) or "").strip()
        if image_id and category:
            image_path = IMAGE_DIR / f"{image_id}.jpg"
            if image_path.exists():
                metadata_rows.append({
                    "id": image_id,
                    "category": category,
                    "path": image_path
                })

if not metadata_rows:
    raise ValueError("No matching image files found. Check IMAGE_DIR and styles.csv.")

category_counts = {}
for row in metadata_rows:
    category_counts[row["category"]] = category_counts.get(row["category"], 0) + 1

sorted_counts = sorted(category_counts.items(), key=lambda item: item[1])

print("Usable images:", len(metadata_rows))
print("Categories:", len(category_counts))
print("\nSmallest categories:")
for category, count in sorted_counts[:20]:
    print(f"{category:30s} {count:6d}")

In [ ]:
# Plot the 20 least represented categories.
least_common = sorted_counts[:20]
labels = [item[0] for item in least_common]
counts = [item[1] for item in least_common]

plt.figure(figsize=(12, 6))
plt.barh(labels, counts)
plt.xlabel("Number of images")
plt.title(f"20 Least Represented Categories — {CATEGORY_COLUMN}")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Use the Phase-2 confusion matrix to choose a target category

The optional CSV should have true classes as rows and predicted classes as columns, with the first column containing row labels. Per-class recall is:

\[
Recall_c = \frac{CM_{c,c}}{\sum_j CM_{c,j}}
\]

If the confusion matrix is unavailable, the notebook selects the least represented category with enough images for a basic GAN experiment. **Review the selected category manually before training.**

In [ ]:
def read_confusion_matrix(path):
    labels = []
    matrix = []

    with Path(path).open("r", newline="", encoding="utf-8") as file:
        reader = csv.reader(file)
        header = next(reader)

        predicted_labels = [value.strip() for value in header[1:]]

        for row in reader:
            if not row:
                continue
            labels.append(row[0].strip())
            matrix.append([float(value) for value in row[1:]])

    if len(labels) != len(predicted_labels):
        raise ValueError("Confusion matrix must be square with matching row/column labels.")

    return labels, predicted_labels, tf.constant(matrix, dtype=tf.float32)


confusion_labels = None
class_recalls = {}

if CONFUSION_MATRIX_PATH.exists():
    true_labels, predicted_labels, confusion_matrix = read_confusion_matrix(
        CONFUSION_MATRIX_PATH
    )

    if true_labels != predicted_labels:
        print("Warning: row and column class ordering differs. Check the CSV labels.")

    row_totals = tf.reduce_sum(confusion_matrix, axis=1)
    diagonal = tf.linalg.diag_part(confusion_matrix)
    recalls = tf.math.divide_no_nan(diagonal, row_totals)

    class_recalls = {
        label: float(recall.numpy())
        for label, recall in zip(true_labels, recalls)
        if float(row_totals[true_labels.index(label)].numpy()) > 0
    }

    print("Per-class recall:")
    for label, recall in sorted(class_recalls.items(), key=lambda item: item[1]):
        print(f"{label:30s} {recall:.3f}")
else:
    print("Phase-2 confusion matrix not found; using category counts as fallback.")

# A practical minimum for this small single-class GAN experiment.
MIN_IMAGES_FOR_GAN = 100

eligible_categories = [
    category for category, count in category_counts.items()
    if count >= MIN_IMAGES_FOR_GAN
]

if not eligible_categories:
    raise ValueError(
        f"No category has at least {MIN_IMAGES_FOR_GAN} images. "
        "Lower MIN_IMAGES_FOR_GAN cautiously or choose a broader category field."
    )

if class_recalls:
    # Map confusion-matrix labels to available dataset categories.
    candidates = [
        category for category in eligible_categories
        if category in class_recalls
    ]
    if candidates:
        target_category = min(candidates, key=lambda c: class_recalls[c])
        selection_reason = "lowest Phase-2 recall among eligible categories"
    else:
        target_category = min(
            eligible_categories, key=lambda c: category_counts[c]
        )
        selection_reason = "least represented eligible category; confusion labels did not match"
else:
    target_category = min(
        eligible_categories, key=lambda c: category_counts[c]
    )
    selection_reason = "least represented eligible category"

print("\nSelected category:", target_category)
print("Image count:", category_counts[target_category])
print("Selection reason:", selection_reason)

## 4. Inspect real examples from the selected category

Before training, verify that the category contains visually consistent product images. A category such as `Shirts` is usually easier to model than a very heterogeneous category.

In [ ]:
target_rows = [
    row for row in metadata_rows
    if row["category"] == target_category
]

print("Selected category images:", len(target_rows))

def load_image(path):
    image_bytes = tf.io.read_file(str(path))
    image = tf.io.decode_jpeg(image_bytes, channels=3)
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    image = tf.cast(image, tf.float32)
    return (image / 127.5) - 1.0

sample_rows = target_rows[:16]
sample_images = [load_image(row["path"]) for row in sample_rows]

plt.figure(figsize=(10, 10))
for index, image in enumerate(sample_images):
    plt.subplot(4, 4, index + 1)
    plt.imshow(tf.cast((image + 1.0) * 127.5, tf.uint8).numpy())
    plt.axis("off")
plt.suptitle(f"Real Examples — {target_category}")
plt.tight_layout()
plt.show()

## 5. Build the training input pipeline

Images are resized to 64×64 and normalized to `[-1, 1]`, matching the generator's `tanh` output.

We shuffle the selected category's images and batch them for GAN training.

In [ ]:
selected_paths = [str(row["path"]) for row in target_rows[:MAX_IMAGES]]

def decode_and_preprocess(path):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image_bytes, channels=3)
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    image = tf.cast(image, tf.float32)
    image = (image / 127.5) - 1.0
    return image

train_dataset = (
    tf.data.Dataset.from_tensor_slices(selected_paths)
    .shuffle(min(len(selected_paths), 2000), seed=RANDOM_SEED)
    .map(decode_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

print("Training images:", len(selected_paths))
print("Batches per epoch:", len(selected_paths) // BATCH_SIZE)

## 6. Build the DCGAN generator and discriminator

**Generator:** starts from a latent noise vector and upsamples it through transposed convolutions to produce a 64×64 RGB image.

**Discriminator:** downsamples a real or generated image and outputs one logit. We use logits with `BinaryCrossentropy(from_logits=True)` for numerical stability.

In [ ]:
def build_generator(latent_dim=LATENT_DIM):
    model = tf.keras.Sequential(name="fashion_generator")
    model.add(tf.keras.layers.Input(shape=(latent_dim,)))

    model.add(tf.keras.layers.Dense(4 * 4 * 512, use_bias=False))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.LeakyReLU(negative_slope=0.2))
    model.add(tf.keras.layers.Reshape((4, 4, 512)))

    for filters in [256, 128, 64]:
        model.add(tf.keras.layers.Conv2DTranspose(
            filters, kernel_size=4, strides=2, padding="same", use_bias=False
        ))
        model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.LeakyReLU(negative_slope=0.2))

    model.add(tf.keras.layers.Conv2DTranspose(
        CHANNELS, kernel_size=4, strides=2, padding="same", activation="tanh"
    ))

    return model


def build_discriminator():
    model = tf.keras.Sequential(name="fashion_discriminator")
    model.add(tf.keras.layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS)))

    for filters in [64, 128, 256, 512]:
        model.add(tf.keras.layers.Conv2D(
            filters, kernel_size=4, strides=2, padding="same"
        ))
        model.add(tf.keras.layers.LeakyReLU(negative_slope=0.2))
        model.add(tf.keras.layers.Dropout(0.3))

    model.add(tf.keras.layers.Flatten())
    model.add(tf.keras.layers.Dense(1))

    return model


generator = build_generator()
discriminator = build_discriminator()

generator.summary()
discriminator.summary()

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_logits, fake_logits):
    real_loss = cross_entropy(tf.ones_like(real_logits), real_logits)
    fake_loss = cross_entropy(tf.zeros_like(fake_logits), fake_logits)
    return real_loss + fake_loss

def generator_loss(fake_logits):
    return cross_entropy(tf.ones_like(fake_logits), fake_logits)


generator_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)

@tf.function
def train_step(real_images):
    noise = tf.random.normal([tf.shape(real_images)[0], LATENT_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_logits = discriminator(real_images, training=True)
        fake_logits = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_logits)
        disc_loss = discriminator_loss(real_logits, fake_logits)

    gen_gradients = gen_tape.gradient(
        gen_loss, generator.trainable_variables
    )
    disc_gradients = disc_tape.gradient(
        disc_loss, discriminator.trainable_variables
    )

    generator_optimizer.apply_gradients(
        zip(gen_gradients, generator.trainable_variables)
    )
    discriminator_optimizer.apply_gradients(
        zip(disc_gradients, discriminator.trainable_variables)
    )

    return gen_loss, disc_loss


print("GAN training step ready.")

## 7. Train the GAN

GAN training can be unstable. Monitor generated samples throughout training rather than relying only on loss values. The fixed noise batch makes visual comparisons across epochs easier.

In [ ]:
fixed_noise = tf.random.normal([16, LATENT_DIM], seed=RANDOM_SEED)

def display_generated_images(model, epoch_label):
    generated = model(fixed_noise, training=False)
    generated = tf.cast((generated + 1.0) * 127.5, tf.uint8)

    plt.figure(figsize=(8, 8))
    for i in range(generated.shape[0]):
        plt.subplot(4, 4, i + 1)
        plt.imshow(generated[i].numpy())
        plt.axis("off")
    plt.suptitle(f"Generated {target_category} — {epoch_label}")
    plt.tight_layout()
    plt.show()


generator_losses = []
discriminator_losses = []

for epoch in range(EPOCHS):
    epoch_gen_losses = []
    epoch_disc_losses = []

    for real_batch in train_dataset:
        gen_loss, disc_loss = train_step(real_batch)
        epoch_gen_losses.append(float(gen_loss.numpy()))
        epoch_disc_losses.append(float(disc_loss.numpy()))

    mean_gen_loss = sum(epoch_gen_losses) / max(len(epoch_gen_losses), 1)
    mean_disc_loss = sum(epoch_disc_losses) / max(len(epoch_disc_losses), 1)

    generator_losses.append(mean_gen_loss)
    discriminator_losses.append(mean_disc_loss)

    print(
        f"Epoch {epoch + 1:03d}/{EPOCHS} | "
        f"G loss: {mean_gen_loss:.4f} | D loss: {mean_disc_loss:.4f}"
    )

    if (epoch + 1) % 10 == 0 or epoch == 0:
        display_generated_images(generator, f"epoch {epoch + 1}")

## 8. Inspect training behaviour and generated diversity

GAN losses do not behave like ordinary supervised learning losses: neither loss needs to decrease monotonically. Look for:
- Product-like shapes and category consistency.
- Different outputs across the grid.
- No repeated identical image or obvious texture collapse.
- No persistent checkerboard artefacts or severe colour distortions.

A generator that produces many near-identical images has likely experienced **mode collapse**.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(generator_losses, label="Generator loss")
plt.plot(discriminator_losses, label="Discriminator loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("DCGAN Training Losses")
plt.legend()
plt.tight_layout()
plt.show()

# Generate a larger grid for visual diversity inspection.
inspection_noise = tf.random.normal([32, LATENT_DIM])
inspection_images = generator(inspection_noise, training=False)
inspection_images = tf.cast((inspection_images + 1.0) * 127.5, tf.uint8)

plt.figure(figsize=(10, 10))
for i in range(32):
    plt.subplot(4, 8, i + 1)
    plt.imshow(inspection_images[i].numpy())
    plt.axis("off")
plt.suptitle(f"Generated Sample Diversity — {target_category}")
plt.tight_layout()
plt.show()

## 9. Export generated images to a review folder

Generated images are **not automatically accepted**. They are saved under:

```text
data/synthetic_fashion/review/<category>/
```

Open and inspect them. Remove unrealistic, duplicated, malformed, or off-category outputs. Copy only approved files into:

```text
data/synthetic_fashion/approved/<category>/
```

This human review step is important: adding low-quality synthetic images can make the classifier worse.

In [ ]:
def safe_name(value):
    return re.sub(r"[^a-zA-Z0-9_-]+", "_", str(value)).strip("_").lower()


category_slug = safe_name(target_category)
category_review_dir = REVIEW_DIR / category_slug
category_approved_dir = APPROVED_DIR / category_slug

category_review_dir.mkdir(parents=True, exist_ok=True)
category_approved_dir.mkdir(parents=True, exist_ok=True)

generated_noise = tf.random.normal([NUM_IMAGES_TO_GENERATE, LATENT_DIM])
generated_batch = generator(generated_noise, training=False)

saved_paths = []
for index in range(NUM_IMAGES_TO_GENERATE):
    image = generated_batch[index]
    image = tf.clip_by_value((image + 1.0) * 127.5, 0, 255)
    image = tf.cast(image, tf.uint8)

    jpeg_bytes = tf.io.encode_jpeg(image, quality=95)
    output_path = category_review_dir / f"synthetic_{index:05d}.jpg"
    tf.io.write_file(str(output_path), jpeg_bytes)
    saved_paths.append(output_path)

print("Generated images saved:", len(saved_paths))
print("Review folder:", category_review_dir)
print("Approved folder (copy reviewed images here):", category_approved_dir)

## 10. Save the generator and training metadata

The generator can be reused later to create more candidate images for this category.

In [ ]:
GENERATOR_PATH = MODEL_DIR / f"fashion_generator_{category_slug}.keras"
GAN_METADATA_PATH = MODEL_DIR / f"fashion_generator_{category_slug}_metadata.pkl"

generator.save(GENERATOR_PATH)

metadata = {
    "category": target_category,
    "category_column": CATEGORY_COLUMN,
    "image_size": IMAGE_SIZE,
    "channels": CHANNELS,
    "latent_dim": LATENT_DIM,
    "training_image_count": len(selected_paths),
    "epochs": EPOCHS
}

with GAN_METADATA_PATH.open("wb") as file:
    pickle.dump(metadata, file)

print("Generator saved:", GENERATOR_PATH)
print("Metadata saved:", GAN_METADATA_PATH)

## 11. Fold approved synthetic images into the classifier training data

The notebook deliberately does not modify the original Fashion Product Images dataset. Once you have manually reviewed the images, copy approved samples into `data/synthetic_fashion/approved/<category>/`.

A downstream Day 6/7 classifier data loader can include these images alongside the original training images for the matching class. Keep the original validation and test sets unchanged.

Example integration contract:

```text
Original training images
        +
Approved synthetic images for target class
        ↓
Day 6/7 classifier training loader
        ↓
Retrain classifier
        ↓
Evaluate on the exact same untouched test set
```

Do not mix synthetic images into validation/test data. Track the number of approved synthetic images and the real-to-synthetic ratio.

In [ ]:
approved_images = list(category_approved_dir.glob("*.jpg")) + list(category_approved_dir.glob("*.jpeg")) + list(category_approved_dir.glob("*.png"))

print("Approved synthetic images:", len(approved_images))
print("Original real images in selected category:", len(target_rows))

if target_rows:
    print(
        "Approved synthetic / real ratio:",
        round(len(approved_images) / len(target_rows), 3)
    )

print("\nNext action:")
print("1. Review files in:", category_review_dir)
print("2. Copy only good samples into:", category_approved_dir)
print("3. Update your Day 6/7 training loader to include approved images.")
print("4. Retrain and compare per-class metrics on the unchanged test set.")

## 12. Weekly Assessment 04 — measure the impact

Run the Day 6/7 classifier twice with the **same train/validation/test split and same evaluation procedure**:

- **Baseline:** real training images only.
- **Augmented:** real training images plus manually approved synthetic images for the target class.

Record per-class precision, recall, F1, and support. Also record overall accuracy and macro-F1. The main question is whether the selected class improved without harming other classes.

Fill in the assessment table after running both experiments:

| Metric | Baseline (real only) | Augmented |
|---|---:|---:|
| Overall accuracy | | |
| Macro precision | | |
| Macro recall | | |
| Macro F1 | | |
| Target-class precision | | |
| Target-class recall | | |
| Target-class F1 | | |

**Do not claim an accuracy or fairness lift unless the evaluation results demonstrate it.** If the target-class recall improves but other classes deteriorate, report that trade-off.

## 13. Day 11 completion checklist

- [ ] Category imbalance inspected.
- [ ] Phase-2 confusion matrix used (or fallback category-count selection documented).
- [ ] Selected category visually inspected.
- [ ] DCGAN trained on the selected category.
- [ ] Generated samples checked for realism and diversity / mode collapse.
- [ ] Candidate images exported to the review folder.
- [ ] Only manually approved images copied to the approved folder.
- [ ] Day 6/7 classifier retrained with approved synthetic images.
- [ ] Per-class metrics compared on the same untouched test set.
- [ ] Weekly Assessment 04 results documented.

### Limitations
1. This is an unconditional, single-category DCGAN baseline; it does not guarantee realistic images.
2. A category with very few and visually diverse examples may not be suitable for GAN training.
3. Attention to visual quality and diversity requires human inspection.
4. Synthetic augmentation can amplify dataset bias; compare per-class metrics and inspect samples.
5. The model generates 64×64 images, so fine product details may be lost.

### Git upload

From the RetailMind project root:

```bash
git add notebooks/day11_gan_image_augmentation.ipynb
git add models/fashion_generator_*.keras
git add models/fashion_generator_*_metadata.pkl
git commit -m "feat: add GAN-based fashion image augmentation"
git push origin main
```

Do not commit the full source dataset or large generated image collections unless your repository's data/versioning policy explicitly allows it.